# 04 — Model Training

Run all 5 models with repeated stratified k-fold CV on mouse and synthetic human data.

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
from src.config import *
from src.data_extraction import load_all_data
from src.preprocessing import preprocess_expression_matrix, prepare_ml_arrays
from src.synthetic_data import generate_synthetic_human_cohort, make_binary_human_dataset
from src.training import run_model_comparison
from src.utils import generate_comparison_table, save_results, describe_results

In [ ]:
data = load_all_data()
matrix = preprocess_expression_matrix(data['mouse_matrix'])
X_mouse, y_mouse, feature_names = prepare_ml_arrays(matrix)
print(f'Mouse data: {X_mouse.shape}, classes: {np.bincount(y_mouse)}')

In [ ]:
print('Training on mouse data (5-fold × 10-repeat CV)...')
results_mouse = run_model_comparison(
    X_mouse, y_mouse, feature_names,
    cv_config={'n_splits': 5, 'n_repeats': 10, 'seed': 42},
    n_stability_iter=200,
)

In [ ]:
describe_results(results_mouse)
save_results(results_mouse, 'mouse_classification_results.csv')
table_mouse = generate_comparison_table(results_mouse)
table_mouse.to_csv(RESULTS_DIR / 'model_comparison_table_mouse.csv', index=False)
table_mouse

In [ ]:
# Synthetic human data
synth = generate_synthetic_human_cohort(data['sig_proteins'])
X_synth, y_synth = make_binary_human_dataset(synth, groups=(0, 2))
X_synth_arr = X_synth.values
feature_names_synth = X_synth.columns.tolist()
print(f'Synthetic human data: {X_synth_arr.shape}, classes: {np.bincount(y_synth)}')

In [ ]:
print('Training on synthetic human data...')
results_synth = run_model_comparison(
    X_synth_arr, y_synth, feature_names_synth,
    cv_config={'n_splits': 5, 'n_repeats': 5, 'seed': 42},
    n_stability_iter=200,
)

In [ ]:
describe_results(results_synth)
save_results(results_synth, 'synthetic_classification_results.csv')
table_synth = generate_comparison_table(results_synth)
table_synth.to_csv(RESULTS_DIR / 'model_comparison_table_synthetic.csv', index=False)
table_synth

In [ ]:
# Real human data (S3 Fig.7: 58 subjects, 2 features)
human = data['human_clinical']
feature_cols_human = [c for c in ['n_APLP1', 'n_CHL1', 'tau_abeta42'] if c in human.columns]
X_human = human[feature_cols_human].dropna().values
y_human = human.loc[human[feature_cols_human].notna().all(axis=1), 'label'].values
print(f'Real human data: {X_human.shape}, classes: {np.bincount(y_human)}')

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
lr = LogisticRegression(penalty='l2', C=1.0, solver='lbfgs', max_iter=5000,
                        class_weight='balanced', random_state=42)
pipe_human = __import__('sklearn.pipeline', fromlist=['Pipeline']).Pipeline([
    ('sc', StandardScaler()), ('clf', lr)
])
aucs = cross_val_score(pipe_human, X_human, y_human, cv=5, scoring='roc_auc')
print(f'Real human AUC (5-fold): {aucs.mean():.3f} ± {aucs.std():.3f}')